In [0]:
from pyspark.sql.functions import col

# 1. Define storage and volume paths
CAPTURE_BASE_PATH = "abfss://bronze@healthcaredatatimi.dfs.core.windows.net/healthcare-streaming/healthcare-readmisssion"
CHECKPOINT_LOCATION = "/Volumes/healthcare_catalog/default/checkpoints/bronze_capture_stream_v2"
SCHEMA_LOCATION = "/Volumes/healthcare_catalog/default/checkpoints/bronze_capture_schema"

# 2. Configure Auto Loader stream (Removed the invalid option)
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "avro")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .load(CAPTURE_BASE_PATH)
)

# 3. Decode Event Hubs binary payload
df_bronze_stream = df_stream.select(
    col("Body").cast("string").alias("raw_payload"),
    col("EnqueuedTimeUtc").alias("event_enqueued_at"),
    col("SequenceNumber"),
    col("Offset")
)

# 4. Write stream to Delta Table
query = (
    df_bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable("healthcare_catalog.default.bronze_clinical_events")
)

In [0]:
%sql
SELECT * FROM healthcare_catalog.default.bronze_clinical_events LIMIT 20;

raw_payload,event_enqueued_at,SequenceNumber,Offset
"{""resourceType"": ""Patient"", ""id"": ""5cbc121b-cd71-4428-b8b7-31e53eba8184"", ""text"": {""status"": ""generated"", ""div"": ""Generated by Synthea.Version identifier: v2.4.0-404-ge7ce2295\n . Person seed: 6457100290386878904 Population seed: 0""}, ""extension"": [{""url"": ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"", ""extension"": [{""url"": ""ombCategory"", ""valueCoding"": {""system"": ""urn:oid:2.16.840.1.113883.6.238"", ""code"": ""2106-3"", ""display"": ""White""}}, {""url"": ""text"", ""valueString"": ""White""}]}, {""url"": ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-ethnicity"", ""extension"": [{""url"": ""ombCategory"", ""valueCoding"": {""system"": ""urn:oid:2.16.840.1.113883.6.238"", ""code"": ""2186-5"", ""display"": ""Not Hispanic or Latino""}}, {""url"": ""text"", ""valueString"": ""Not Hispanic or Latino""}]}, {""url"": ""http://hl7.org/fhir/StructureDefinition/patient-mothersMaidenName"", ""valueString"": ""Deadra347 Borer986""}, {""url"": ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-birthsex"", ""valueCode"": ""M""}, {""url"": ""http://hl7.org/fhir/StructureDefinition/patient-birthPlace"", ""valueAddress"": {""city"": ""Billerica"", ""state"": ""Massachusetts"", ""country"": ""US""}}, {""url"": ""http://synthetichealth.github.io/synthea/disability-adjusted-life-years"", ""valueDecimal"": 14.062655945052095}, {""url"": ""http://synthetichealth.github.io/synthea/quality-adjusted-life-years"", ""valueDecimal"": 58.93734405494791}], ""identifier"": [{""system"": ""https://github.com/synthetichealth/synthea"", ""value"": ""2fa15bc7-8866-461a-9000-f739e425860a""}, {""type"": {""coding"": [{""system"": ""http://terminology.hl7.org/CodeSystem/v2-0203"", ""code"": ""MR"", ""display"": ""Medical Record Number""}], ""text"": ""Medical Record Number""}, ""system"": ""http://hospital.smarthealthit.org"", ""value"": ""2fa15bc7-8866-461a-9000-f739e425860a""}, {""type"": {""coding"": [{""system"": ""http://terminology.hl7.org/CodeSystem/v2-0203"", ""code"": ""SS"", ""display"": ""Social Security Number""}], ""text"": ""Social Security Number""}, ""system"": ""http://hl7.org/fhir/sid/us-ssn"", ""value"": ""999-93-7537""}, {""type"": {""coding"": [{""system"": ""http://terminology.hl7.org/CodeSystem/v2-0203"", ""code"": ""DL"", ""display"": ""Driver's License""}], ""text"": ""Driver's License""}, ""system"": ""urn:oid:2.16.840.1.113883.4.3.25"", ""value"": ""S99948707""}, {""type"": {""coding"": [{""system"": ""http://terminology.hl7.org/CodeSystem/v2-0203"", ""code"": ""PPN"", ""display"": ""Passport Number""}], ""text"": ""Passport Number""}, ""system"": ""http://standardhealthrecord.org/fhir/StructureDefinition/passportNumber"", ""value"": ""X14078167X""}], ""name"": [{""use"": ""official"", ""family"": ""Brekke496"", ""given"": [""Aaron697""], ""prefix"": [""Mr.""]}], ""telecom"": [{""system"": ""phone"", ""value"": ""555-677-3119"", ""use"": ""home""}], ""gender"": ""male"", ""birthDate"": ""1945-12-10"", ""address"": [{""extension"": [{""url"": ""http://hl7.org/fhir/StructureDefinition/geolocation"", ""extension"": [{""url"": ""latitude"", ""valueDecimal"": 41.93879298871088}, {""url"": ""longitude"", ""valueDecimal"": -71.06682353144593}]}], ""line"": [""894 Brakus Bypass""], ""city"": ""Taunton"", ""state"": ""Massachusetts"", ""postalCode"": ""02718"", ""country"": ""US""}], ""maritalStatus"": {""coding"": [{""system"": ""http://terminology.hl7.org/CodeSystem/v3-MaritalStatus"", ""code"": ""S"", ""display"": ""S""}], ""text"": ""S""}, ""multipleBirthBoolean"": false, ""communication"": [{""language"": {""coding"": [{""system"": ""urn:ietf:bcp:47"", ""code"": ""en-US"", ""display"": ""English""}], ""text"": ""English""}}]}",7/21/2026 1:25:44 PM,7502,8589934592
"{""resourceType"": ""Encounter"", ""id"": ""f78d73fc-9f9b-46d5-93aa-f5db86ba914c"", ""status"": ""finished"", ""class"": {""system"": ""http://terminology.h